# CNN Hyperparameter Tuning — Mel Spectrogram Input

Uses **Optuna** to search over mel spectrogram input parameters.  
Objective: **maximize unseen-environment accuracy**.

All shared classes are imported from `cnn_shared.py` — no duplication.

**Parameters tuned (mel spectrogram):**
| Parameter | What it controls |
|-----------|------------------|
| `n_mels` | Mel frequency bins — higher = finer frequency detail |
| `hop_length` | Time step in samples — lower = finer time resolution |
| `n_fft` | FFT window size — larger = finer freq, coarser time |
| `win_length` | Analysis window length — shorter = better transient capture |
| `f_min` | Lowest frequency bound (Hz) |
| `f_max` | Highest frequency bound (Hz) |
| `power` | `1.0` = magnitude spectrogram, `2.0` = power spectrogram |

**Commented out (not tuned here — restore to re-enable):**
- `snr_min` / `snr_max`, `time_shift_ratio`, `freq_mask_param`, `time_mask_param`, `vol_reduction_prob`
- `dropout_conv`, `dropout_fc`


### Install Optuna (if needed)

In [ ]:
import importlib, subprocess, sys
if importlib.util.find_spec('optuna') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'optuna'])
print('optuna ready')

### Imports

In [ ]:
import sys, os

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    repo_dir = '/content/keyboard_sound'
    if not os.path.exists(repo_dir):
        !git clone https://github.com/ayushma18/keyboard_sound {repo_dir}
    %cd {repo_dir}
    !pip install -r requirements_colab.txt

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchaudio
import matplotlib.pyplot as plt
import optuna
from optuna.pruners import MedianPruner
from optuna.samplers import TPESampler
from torch.utils.data import DataLoader
from torchvision.transforms import Compose, ToTensor
from torchaudio.transforms import FrequencyMasking, TimeMasking
from sklearn.model_selection import train_test_split

# ── Import everything shared from cnn_shared.py ───────────────────────────────
from cnn_shared import (
    NoiseLibrary, AggressiveMultiVariationAugmentation, TimeShifting,
    AudioDataset, TrainingDataset,
    CNN, ConvBlock, init_weights, get_device,
    MelConfig, build_mel_transform,
    calculate_rms, add_noise_snr, add_white_noise_snr, add_gaussian_noise_snr, reduce_volume_db,
)

print('All imports OK')


### Paths

In [ ]:
data_path        = 'Data/combined/AULA-NUM'
unseen_data_path = 'Data/combined/test-num'
noise_path       = 'model/Noises'
tuned_model_path = f"model/CNN-Tuned-{data_path.split('/')[-1]}.pkl"

### Device & mel transform

In [ ]:
device      = get_device()
sample_rate = 44100

# Default mel config (CNN.ipynb baseline) shown here for reference.
# Each Optuna trial will build its own transform from MEL_SEARCH_SPACE.
_default_mel = MelConfig()   # n_mels=128, hop=256, n_fft=1024, win=512
base_tf      = build_mel_transform(_default_mel)

print(f'Device      : {device}')
print(f'Default mel : {_default_mel.label()}')


### Load Noise Library

In [ ]:
noise_library = NoiseLibrary(noise_path)
print('Noise library ready')

### Load Data & Split

In [ ]:
dataset        = AudioDataset(data_path)
unseen_dataset = AudioDataset(unseen_data_path)
NUM_CLASSES    = dataset.num_classes()

targets = [d[1].item() for d in dataset.dataset]
train_idx, tmp_idx = train_test_split(range(len(dataset)), test_size=0.3, stratify=targets)
val_idx,  test_idx = train_test_split(tmp_idx, test_size=0.33, stratify=[targets[i] for i in tmp_idx])

init_train = torch.utils.data.Subset(dataset, train_idx)
init_val   = torch.utils.data.Subset(dataset, val_idx)
init_test  = torch.utils.data.Subset(dataset, test_idx)

print(f'Classes: {NUM_CLASSES}  |  Train: {len(init_train)}  Val: {len(init_val)}  Test: {len(init_test)}  Unseen: {len(unseen_dataset)}')

### Tuning Configuration

Edit `MEL_SEARCH_SPACE` to widen or narrow the search ranges.  
Augmentation and dropout search spaces are commented out below.


In [ ]:
TUNING_N_TRIALS   = 30   # total Optuna trials
TUNING_MAX_EPOCHS = 80   # epochs per trial (short run; best params used for full retrain)
TUNING_BATCH_SIZE = 32

# ── Mel spectrogram search space ──────────────────────────────────────────────
# Categorical lists for n_mels / hop_length / n_fft / win_length / power so that
# win_length can be clamped to ≤ n_fft inside the objective.
MEL_SEARCH_SPACE = {
    'n_mels':     [64, 128, 256],        # frequency bins
    'hop_length': [128, 256, 300],       # time step (lower = finer time grid)
    'n_fft':      [512, 1024, 2048],     # FFT window
    'win_length': [256, 512, 1024],      # analysis window (clamped to ≤ n_fft in objective)
    'f_min':      (20.0,  300.0),        # float range (Hz)
    'f_max':      (6000.0, 20000.0),     # float range (Hz)
    'power':      [1.0, 2.0],            # 1.0=magnitude, 2.0=power
}

# ── Commented out: augmentation + model dropout parameters ────────────────────
# Uncomment and merge into the objective to tune these alongside mel params.
#
# AUG_SEARCH_SPACE = {
#     'snr_min':             (3.0,  15.0),
#     'snr_max':             (20.0, 35.0),
#     'time_shift_ratio':    (0.1,  0.6),
#     'freq_mask_param':     (0,    20),    # int; 0 = disabled
#     'time_mask_param':     (0,    20),    # int; 0 = disabled
#     'vol_reduction_prob':  (0.1,  0.6),
#     'dropout_conv':        (0.1,  0.4),
#     'dropout_fc':          (0.3,  0.6),
# }

print('Mel spectrogram search space:')
for k, v in MEL_SEARCH_SPACE.items():
    print(f'  {k:12s}: {v}')


### Optuna Objective (Mel Spectrogram Search)


In [ ]:
def objective(trial):
    # ── Suggest mel spectrogram parameters ────────────────────────────────────
    n_mels     = trial.suggest_categorical('n_mels',     MEL_SEARCH_SPACE['n_mels'])
    hop_length = trial.suggest_categorical('hop_length', MEL_SEARCH_SPACE['hop_length'])
    n_fft      = trial.suggest_categorical('n_fft',      MEL_SEARCH_SPACE['n_fft'])
    # Clamp win_length to ≤ n_fft to satisfy torchaudio constraint
    win_length = min(trial.suggest_categorical('win_length', MEL_SEARCH_SPACE['win_length']), n_fft)
    f_min      = trial.suggest_float('f_min', *MEL_SEARCH_SPACE['f_min'])
    f_max      = trial.suggest_float('f_max', *MEL_SEARCH_SPACE['f_max'])
    power      = trial.suggest_categorical('power', MEL_SEARCH_SPACE['power'])

    # Prune invalid frequency ranges
    if f_max <= f_min + 500:
        raise optuna.TrialPruned()

    mel_cfg  = MelConfig(n_mels=n_mels, hop_length=hop_length, n_fft=n_fft,
                         win_length=win_length, f_min=f_min, f_max=f_max, power=power)
    trial_tf = build_mel_transform(mel_cfg)

    # Fixed augmentation (augmentation params not being tuned in this notebook)
    # To tune these too, move them to MEL_SEARCH_SPACE and suggest them here.
    aug = AggressiveMultiVariationAugmentation(
        noise_library=noise_library,
        snr_range=(5, 30),
        volume_reduction_prob=0.4,
        volume_reduction_range=(2, 15),
        multi_aug_prob=0.3,
    )
    aug_tf = Compose([aug, TimeShifting(0.5), trial_tf])

    # num_workers=0 required inside Jupyter to avoid DataLoader deadlocks
    dl_kw     = dict(batch_size=TUNING_BATCH_SIZE, num_workers=0)
    train_dl  = DataLoader(TrainingDataset(init_train,     aug_tf),    shuffle=True,  **dl_kw)
    val_dl    = DataLoader(TrainingDataset(init_val,       trial_tf),  shuffle=False, **dl_kw)
    unseen_dl = DataLoader(TrainingDataset(unseen_dataset, trial_tf),  shuffle=False, **dl_kw)

    # Fixed dropout (not being tuned here)
    model = CNN(num_classes=NUM_CLASSES, dropout_conv=0.25, dropout_fc=0.5)
    model.apply(init_weights)
    model.to(device)

    opt       = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
    sched     = optim.lr_scheduler.ReduceLROnPlateau(opt, mode='min', factor=0.5, patience=10)
    criterion = nn.CrossEntropyLoss()

    best_unseen_acc = 0.0

    for epoch in range(TUNING_MAX_EPOCHS):
        model.train()
        for inputs, labels in train_dl:
            inputs, labels = inputs.to(device), torch.squeeze(labels).to(device)
            opt.zero_grad()
            loss = criterion(model(inputs), labels)
            loss.backward()
            opt.step()

        model.eval()
        with torch.no_grad():
            val_loss = 0.0
            for inputs, labels in val_dl:
                inputs, labels = inputs.to(device), torch.squeeze(labels).to(device)
                val_loss += criterion(model(inputs), labels).item()
            sched.step(val_loss / len(val_dl))

            correct = total = 0
            for inputs, labels in unseen_dl:
                inputs, labels = inputs.to(device), torch.squeeze(labels).to(device)
                _, pred = torch.max(model(inputs), 1)
                correct += (pred == labels).sum().item()
                total   += labels.size(0)

        unseen_acc      = correct / total
        best_unseen_acc = max(best_unseen_acc, unseen_acc)

        trial.report(unseen_acc, epoch)
        if trial.should_prune():
            raise optuna.TrialPruned()

    return best_unseen_acc


### Run Optuna Study

In [ ]:
optuna.logging.set_verbosity(optuna.logging.WARNING)

study = optuna.create_study(
    direction='maximize',
    sampler=TPESampler(seed=42),
    pruner=MedianPruner(n_startup_trials=5, n_warmup_steps=20),
)

print(f'Starting Optuna search: {TUNING_N_TRIALS} trials x {TUNING_MAX_EPOCHS} epochs each')
print(f'Objective: maximize unseen-environment accuracy\n')

study.optimize(objective, n_trials=TUNING_N_TRIALS, show_progress_bar=True)

print('\n' + '='*60)
print('TUNING COMPLETE')
print('='*60)
print(f'Best unseen accuracy : {study.best_value:.4f}')
print('Best hyperparameters :')
for k, v in study.best_params.items():
    print(f'  {k:25s} = {v}')

### Trial History Plot

In [ ]:
completed   = [t for t in study.trials if t.value is not None]
values      = [t.value for t in completed]
best_so_far = [max(values[:i+1]) for i in range(len(values))]

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.scatter(range(len(values)), values, s=20, alpha=0.7, label='Trial unseen acc')
plt.plot(best_so_far, color='red', label='Best so far')
plt.xlabel('Trial'); plt.ylabel('Unseen accuracy')
plt.title('Optuna trial history'); plt.legend()

try:
    importances = optuna.importance.get_param_importances(study)
    plt.subplot(1, 2, 2)
    names = list(importances.keys())
    imps  = list(importances.values())
    plt.barh(names[::-1], imps[::-1])
    plt.xlabel('Importance'); plt.title('Hyperparameter importance')
except Exception:
    pass

plt.tight_layout(); plt.show()

### Full Retrain with Best Hyperparameters

In [ ]:
bp = study.best_params

FULL_MAX_EPOCHS          = 1200
FULL_EARLY_STOP_PATIENCE = 50
FULL_EARLY_STOP_DELTA    = 0.001

# Reconstruct the best mel config from Optuna best params
best_mel_cfg = MelConfig(
    n_mels=bp['n_mels'],
    hop_length=bp['hop_length'],
    n_fft=bp['n_fft'],
    win_length=min(bp['win_length'], bp['n_fft']),
    f_min=bp['f_min'],
    f_max=bp['f_max'],
    power=bp['power'],
)
best_tf = build_mel_transform(best_mel_cfg)

print(f'Best mel config : {best_mel_cfg.label()}')

best_aug = AggressiveMultiVariationAugmentation(
    noise_library=noise_library,
    snr_range=(5, 30),
    volume_reduction_prob=0.4,
    volume_reduction_range=(2, 15),
    multi_aug_prob=0.3,
)
best_aug_tf = Compose([best_aug, TimeShifting(0.5), best_tf])

dl_kw     = dict(batch_size=32, num_workers=0)
train_dl  = DataLoader(TrainingDataset(init_train,     best_aug_tf), shuffle=True,  **dl_kw)
val_dl    = DataLoader(TrainingDataset(init_val,       best_tf),     shuffle=False, **dl_kw)
test_dl   = DataLoader(TrainingDataset(init_test,      best_tf),     shuffle=False, **dl_kw)
unseen_dl = DataLoader(TrainingDataset(unseen_dataset, best_tf),     shuffle=False, **dl_kw)

model     = CNN(num_classes=NUM_CLASSES, dropout_conv=0.25, dropout_fc=0.5)
model.apply(init_weights)
model.to(device)

optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=10)
criterion = nn.CrossEntropyLoss()

print(f'\nFull retrain — up to {FULL_MAX_EPOCHS} epochs with best mel config')


In [ ]:
train_losses, train_accs = [], []
val_losses,   val_accs   = [], []
unseen_accs  = []
best_val_acc = 0.0
no_improve   = 0

for epoch in range(FULL_MAX_EPOCHS):
    model.train()
    epoch_loss = correct = total = 0
    for inputs, labels in train_dl:
        inputs, labels = inputs.to(device), torch.squeeze(labels).to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        _, pred  = torch.max(outputs, 1)
        correct += (pred == labels).sum().item()
        total   += labels.size(0)
        epoch_loss += loss.item()

    train_losses.append(epoch_loss / len(train_dl))
    train_accs.append(correct / total)

    model.eval()
    with torch.no_grad():
        val_loss = val_c = val_t = 0
        for inputs, labels in val_dl:
            inputs, labels = inputs.to(device), torch.squeeze(labels).to(device)
            outputs  = model(inputs)
            val_loss += criterion(outputs, labels).item()
            _, pred  = torch.max(outputs, 1)
            val_c   += (pred == labels).sum().item()
            val_t   += labels.size(0)

        avg_val_loss = val_loss / len(val_dl)
        val_acc      = val_c / val_t
        val_losses.append(avg_val_loss)
        val_accs.append(val_acc)

        u_c = u_t = 0
        for inputs, labels in unseen_dl:
            inputs, labels = inputs.to(device), torch.squeeze(labels).to(device)
            _, pred = torch.max(model(inputs), 1)
            u_c += (pred == labels).sum().item()
            u_t += labels.size(0)
        unseen_acc = u_c / u_t
        unseen_accs.append(unseen_acc)

    scheduler.step(avg_val_loss)

    print(f'Epoch [{epoch+1}/{FULL_MAX_EPOCHS}]  '
          f'Train {train_accs[-1]:.4f}  Val {val_acc:.4f}  Unseen {unseen_acc:.4f}')

    if val_acc > best_val_acc + FULL_EARLY_STOP_DELTA:
        best_val_acc = val_acc
        no_improve   = 0
        torch.save(model.state_dict(), tuned_model_path)
        print(f'  Saved best model (val acc {val_acc:.4f})')
    else:
        no_improve += 1

    if no_improve >= FULL_EARLY_STOP_PATIENCE:
        print(f'\nEarly stopping at epoch {epoch+1}')
        break

    if (epoch + 1) % 20 == 0:
        plt.figure(figsize=(12, 4))
        plt.subplot(1,2,1); plt.title('Loss')
        plt.plot(train_losses, label='train'); plt.plot(val_losses, label='val'); plt.legend()
        plt.subplot(1,2,2); plt.title('Accuracy')
        plt.plot(train_accs, label='train'); plt.plot(val_accs, label='val')
        plt.plot(unseen_accs, '--', label='unseen'); plt.legend()
        plt.tight_layout(); plt.show()

print(f'\nDone.  Best val acc: {best_val_acc:.4f}  |  Final unseen acc: {unseen_accs[-1]:.4f}')
print(f'Model saved to: {tuned_model_path}')

### Final Results

In [ ]:
plt.figure(figsize=(12, 4))
plt.subplot(1,2,1); plt.title('Loss (full retrain)')
plt.plot(train_losses, label='train'); plt.plot(val_losses, label='val'); plt.legend()
plt.subplot(1,2,2); plt.title('Accuracy (full retrain)')
plt.plot(train_accs, label='train'); plt.plot(val_accs, label='val')
plt.plot(unseen_accs, '--', color='orange', label='unseen'); plt.legend()
plt.tight_layout(); plt.show()

print(f'Best val acc    : {max(val_accs):.4f}')
print(f'Best unseen acc : {max(unseen_accs):.4f}')
print(f'Final unseen acc: {unseen_accs[-1]:.4f}')
print(f'Epochs trained  : {len(val_accs)}')